# Multi-period Optimal Power Flow (MPOPF)
---



The multiperiod ACOPF model from potpourri is imported to set up and solve the multiperiod optimal power flow problem.


In [14]:
import simbench as sb
import pyomo.environ as pe
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [15]:
from potpourri.models_multi_period.ACOPF_multi_period import ACOPF_multi_period

1. Load network data


In [16]:
net = sb.get_simbench_net("1-LV-urban6--0-sw")


2. Set up optimization horizon

The optimization horizon spans from a starting time (fromT) to an ending time (toT). In this case, we set the horizon to 96 time steps (1 day) with 15-minute intervals. The standard timeseries data used for load and generation elements in the grid are the Simbench profiles for each grid element.


In [17]:
fromT = 10000
toT = fromT + 12 # 1 day for 15 min intervals

3. Set Generator Power Limits and Run Power Flow

We then set the power limits for each generator (max_p_mw and min_p_mw) and mark them as controllable.

In [18]:
net.sgen["max_p_mw"] = net.sgen["p_mw"]
net.sgen["min_p_mw"] = net.sgen["p_mw"]
net.sgen['controllable'] = True


4. Set up ACOPF Model

Now, we instantiate the multiperiod ACOPF model from potpourri and add the optimization problem. The objective function can be set seperately. In this case, we are minimizing the voltage deviation.

In [19]:
opf = ACOPF_multi_period(net, toT, fromT, num_vehicles=5)
opf.add_OPF()
opf.add_arbitrage_objective()

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)



Creating Model at:  Fri May  2 18:21:31 2025


4. Solve Multi-period OPF

We solve the ACOPF optimization problem using the ipopt solver. The solver adjusts the power generation and reactive power over the entire time horizon to minimize the objective function while satisfying the grid constraints. After solving the OPF problem, we access some results, such as the power generation of an exemplary generator over the entire optimiation horizon.


In [20]:
opf.solve(print_solver_output=False)

# example of how to access the results -> sg output throughout the time horizon
for t in opf.model.T:
    print(pe.value(opf.model.psG[0, t]))


Solved successfully
1.4302282864278935e-39
1.304404339507164e-41
4.198710424617783e-42
4.183575860462611e-43
2.513038311642291e-43
2.776467114598453e-45
6.181669526884784e-45
2.3049016618872338e-45
5.704246875391454e-47
3.7886938558253246e-47
0.0
1.725618887286162e-47


Conclusion

In this notebook, we have demonstrated how to use the potpourri package to solve a Multi-Period Optimal Power Flow (MP-ACOPF) problem. We started by loading network data from Simbench, set up the multiperiod ACOPF optimization horizon and optimization problem, solved it using Pyomo, and analyzed the results. This workflow can be extended to other power system models and optimization objectives.

